<p align="center">
  <img src="../assets/prodinno_logo.png" alt="Prodinno" width="200">
</p>

<h4 align="center">Session 1 · Linear & Logistic Regression</h4>
<h1 align="center">Linear Regression — Avocado Prices</h1>
<p align="center"><i>Turning cleaned, understood data into a modeling-ready feature matrix</i></p>

---

## Notebook Roadmap

EDA told us what's in the data and what to watch out for. Now we make a series of **explicit,
justified** decisions to turn `avocado_clean.csv` into two matrices a linear regression model
can actually consume:

1. Parse `Date` into `year`, `month`, and `quarter` features.
2. Encode `type` as a binary indicator.
3. Resolve the `region` aggregate/granular hierarchy -- a data-integrity decision, not a
   mechanical one.
4. One-hot encode the remaining, granular `region` values.
5. Scale numeric features.
6. Split into train and test sets, with a discussion of why a random split is a simplification.
7. Save `avocado_processed_train.csv` and `avocado_processed_test.csv`.

Every step below is a decision with a rationale attached -- that rationale is the part worth
remembering, more than the exact code.


In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)

RANDOM_SEED = 42
DATA_DIR = Path("data")

df = pd.read_csv(DATA_DIR / "avocado_clean.csv", parse_dates=["Date"])
print(f"Shape: {df.shape}")
df.head()


Shape: (18244, 13)


,Date,AveragePrice,Total Volume,4046,4225,4770,Total Bags,Small Bags,Large Bags,XLarge Bags,type,year,region
0,2015-12-27,1.33,64236.62,1036.74,54454.85,48.16,8696.87,8603.62,93.25,0.0,conventional,2015,Albany
1,2015-12-20,1.35,54876.98,674.28,44638.81,58.33,9505.56,9408.07,97.49,0.0,conventional,2015,Albany
2,2015-12-13,0.93,118220.22,794.70,109149.67,130.50,8145.35,8042.21,103.14,0.0,conventional,2015,Albany
3,2015-12-06,1.08,78992.15,1132.00,71976.41,72.58,5811.16,5677.40,133.76,0.0,conventional,2015,Albany
4,2015-11-29,1.28,51039.60,941.48,43838.39,75.78,6183.95,5986.26,197.69,0.0,conventional,2015,Albany


## 1. Date Features

`Date` itself is not a number a linear model can use directly, but the calendar signal we saw
in EDA (trend + seasonality) is real and worth capturing explicitly. We derive three simple
features:

- `year` -- captures the multi-year trend/level shift we saw in the time-series plot.
- `month` -- captures within-year seasonality (harvest cycles, holiday demand, etc.).
- `quarter` -- a coarser seasonal bucket that's often more stable to encode as one-hot than
  12 individual months, since each quarter still gets a healthy number of observations.

We keep `year` and `month` as ordinal numeric features (there is a real, meaningful order and
roughly-linear spacing between them) and one-hot encode `quarter` (4 categories -- cheap, and
lets the model learn a non-linear seasonal bump per quarter rather than assuming a strictly
linear effect within the year).


In [2]:
df["year"] = df["Date"].dt.year
df["month"] = df["Date"].dt.month
df["quarter"] = df["Date"].dt.quarter

df[["Date", "year", "month", "quarter"]].head()


,Date,year,month,quarter
0,2015-12-27,2015,12,4
1,2015-12-20,2015,12,4
2,2015-12-13,2015,12,4
3,2015-12-06,2015,12,4
4,2015-11-29,2015,11,4


`quarter` is a categorical seasonal bucket rather than a continuous quantity -- there is no
reason to assume the effect of "being in Q4" is exactly 2x the effect of "being in Q2" just
because 4 is 2x2. We one-hot encode it here (4 categories, `drop_first=True` to avoid the
dummy variable trap discussed below), rather than treating it as ordinal like `year`/`month`.


In [3]:
df = pd.get_dummies(df, columns=["quarter"], prefix="quarter", drop_first=True)
quarter_cols = [c for c in df.columns if c.startswith("quarter_")]
print(f"Created {len(quarter_cols)} quarter dummy columns: {quarter_cols}")


Created 3 quarter dummy columns: ['quarter_2', 'quarter_3', 'quarter_4']


## 2. Encoding `type`

`type` has exactly two categories (`conventional`, `organic`), so a single binary column is
sufficient -- encoding it as two separate one-hot columns would just be redundant (each is the
exact complement of the other).


In [4]:
df["type_organic"] = (df["type"] == "organic").astype(int)
df[["type", "type_organic"]].drop_duplicates()


,type,type_organic
0,conventional,0
9123,organic,1


## 3. Resolving the `region` Hierarchy -- a Data-Integrity Decision

This is the moment we act on the gotcha flagged in EDA. Recall: 11 of the 54 `region` values
are **aggregate rollups** (`TotalUS`, `California`, and nine other multi-state groupings) that
already contain the volume of other rows in the same column. If we one-hot encode all 54
values and hand them to the model as-is, we are not giving it 54 independent markets -- we are
giving it a mix of genuine markets and arithmetic restatements of those same markets, with no
way for the model to know the difference.

**The decision:** keep only the granular (city/state-level) regions, and drop the aggregate
rollups entirely before encoding. This is deliberately the simplest defensible choice
available to us:

- It removes the double-counting at its source, rather than trying to down-weight or
  reconcile it after the fact.
- It costs us relatively little -- we are dropping 11 rows-worth of *region categories*, not
  11% of our *data volume*; every week/type combination for every granular region is still
  fully represented.
- The alternative -- keeping only the aggregates and dropping the granular rows -- would throw
  away the finer-grained regional signal that Section 3 of the EDA notebook showed genuinely
  matters, so we reject it.

> **This is a judgment call, and we are naming it as one.** A different, equally defensible
> project could instead choose to model at the aggregate level only (fewer, more stable
> categories, more historical precedent in retail-economics literature). We are choosing
> granularity because our modeling goal is to explain *local* price variation, and because we
> have enough data per granular region for a one-hot column to be well-estimated.


In [5]:
AGGREGATE_REGIONS = [
    "TotalUS", "California", "GreatLakes", "Midsouth", "Northeast",
    "NorthernNewEngland", "Plains", "SouthCentral", "Southeast",
    "West", "WestTexNewMexico",
]

before_rows = len(df)
df = df[~df["region"].isin(AGGREGATE_REGIONS)].reset_index(drop=True)
after_rows = len(df)

print(f"Rows before dropping aggregate regions: {before_rows}")
print(f"Rows after dropping aggregate regions:  {after_rows}  (removed {before_rows - after_rows})")
print(f"Remaining distinct regions: {df['region'].nunique()}")


Rows before dropping aggregate regions: 18244
Rows after dropping aggregate regions:  14530  (removed 3714)
Remaining distinct regions: 43


## 4. One-Hot Encoding `region`

With only granular regions left, one-hot encoding is now safe -- each resulting column
represents one non-overlapping market. We drop one category (`drop_first=True`) to avoid the
"dummy variable trap": with all $k$ categories included plus an intercept, the columns would
be perfectly collinear (they always sum to 1), which makes the design matrix singular for
exact OLS. Dropping one category folds it into the intercept as the implicit baseline.


In [6]:
df = pd.get_dummies(df, columns=["region"], prefix="region", drop_first=True)
region_cols = [c for c in df.columns if c.startswith("region_")]
print(f"Created {len(region_cols)} region dummy columns.")
df[region_cols].head(3)


Created 42 region dummy columns.


,region_Atlanta,region_BaltimoreWashington,region_Boise,region_Boston,region_BuffaloRochester,region_Charlotte,region_Chicago,region_CincinnatiDayton,region_Columbus,region_DallasFtWorth,region_Denver,region_Detroit,region_GrandRapids,region_HarrisburgScranton,region_HartfordSpringfield,region_Houston,region_Indianapolis,region_Jacksonville,region_LasVegas,region_LosAngeles,region_Louisville,region_MiamiFtLauderdale,region_Nashville,region_NewOrleansMobile,region_NewYork,region_Orlando,region_Philadelphia,region_PhoenixTucson,region_Pittsburgh,region_Portland,region_RaleighGreensboro,region_RichmondNorfolk,region_Roanoke,region_Sacramento,region_SanDiego,region_SanFrancisco,region_Seattle,region_SouthCarolina,region_Spokane,region_StLouis,region_Syracuse,region_Tampa
0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False


## 5. Building the Final Feature Set and Splitting

We now assemble the modeling frame: drop columns we no longer need in their raw form
(`Date`, the original `type` string, `Total Bags`-decomposed columns we're keeping as-is for
now since VIF/multicollinearity handling is a notebook-03 topic, not a preprocessing-removal
topic), and separate `AveragePrice` as the target.

We split **80/20 into train and test** using a plain random split.

> **A honest caveat about the split:** this data has real time structure (we saw the trend in
> EDA). A methodologically stricter approach for a true forecasting use case would split by
> **date** -- e.g., train on 2015-2017, test on 2018 -- so the model is only ever evaluated on
> genuinely *future* data it could not have seen a whisper of during training. A random split
> lets information from "the future" (e.g., a 2018 row) sit in the training set while a 2016
> row from the same region sits in the test set, which is optimistic compared to a real
> deployment scenario. We use a random split here because our goal in this workshop is to
> teach the mechanics of linear regression evaluation cleanly, not to build a production
> forecasting pipeline -- but in a real forecasting project, this would be the first thing to
> change.


In [7]:
drop_cols = ["Date", "type"]
model_df = df.drop(columns=drop_cols)

target_col = "AveragePrice"
X = model_df.drop(columns=[target_col])
y = model_df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED
)

print(f"X_train: {X_train.shape}   X_test: {X_test.shape}")


X_train: (11624, 56)   X_test: (2906, 56)


## 6. Scaling Numeric Features

Linear regression itself does not strictly *require* scaled inputs to fit correctly, but
scaling makes the fitted coefficients comparable to each other in magnitude, and it is good
practice for any later regularized variant (Ridge/Lasso) that penalizes coefficient size
directly. We scale only the genuinely continuous numeric columns -- **not** the one-hot
columns (which are already on a meaningful 0/1 scale) and **not** the target.

Critically, we **fit the scaler on the training data only** and apply that same fit to the
test data. Fitting on the full dataset (train + test combined) would leak information about
the test set's distribution into preprocessing -- a subtle form of data leakage.


In [8]:
numeric_features = [
    "Total Volume", "4046", "4225", "4770",
    "Total Bags", "Small Bags", "Large Bags", "XLarge Bags",
    "year", "month",
]

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_test_scaled[numeric_features] = scaler.transform(X_test[numeric_features])

X_train_scaled[numeric_features].describe().round(3)


,Total Volume,4046,4225,4770,Total Bags,Small Bags,Large Bags,XLarge Bags,year,month
count,11624.000,11624.000,11624.000,11624.000,11624.000,11624.000,11624.000,11624.000,11624.000,11624.000
mean,-0.000,0.000,-0.000,0.000,-0.000,0.000,-0.000,-0.000,0.000,0.000
std,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000
min,-0.562,-0.401,-0.525,-0.341,-0.495,-0.436,-0.374,-0.236,-1.229,-1.473
25%,-0.543,-0.399,-0.513,-0.341,-0.471,-0.423,-0.373,-0.236,-1.229,-0.907
50%,-0.416,-0.378,-0.425,-0.336,-0.333,-0.311,-0.335,-0.236,-0.164,-0.059
75%,0.136,-0.012,0.018,-0.185,0.059,0.057,-0.113,-0.225,0.900,0.790
max,11.991,14.468,13.199,13.627,18.077,20.457,16.607,15.410,1.964,1.638


> **Sanity check:** after scaling, every numeric feature in the *training* set has mean
> ~0 and standard deviation ~1, by construction. The test set will be close but not exactly
> 0/1 -- which is expected and correct, since the test set's own mean/std were never used to
> fit the scaler.


## 7. Saving the Processed Datasets

We recombine the scaled features with their targets and write both splits to disk. Every
downstream notebook will load these two files directly rather than repeating any of the
feature engineering above.


In [9]:
train_out = X_train_scaled.copy()
train_out[target_col] = y_train.values

test_out = X_test_scaled.copy()
test_out[target_col] = y_test.values

train_out.to_csv(DATA_DIR / "avocado_processed_train.csv", index=False)
test_out.to_csv(DATA_DIR / "avocado_processed_test.csv", index=False)

print(f"Saved avocado_processed_train.csv -- shape {train_out.shape}")
print(f"Saved avocado_processed_test.csv  -- shape {test_out.shape}")
train_out.head()


Saved avocado_processed_train.csv -- shape (11624, 57)
Saved avocado_processed_test.csv  -- shape (2906, 57)


,Total Volume,4046,4225,4770,Total Bags,Small Bags,Large Bags,XLarge Bags,year,month,quarter_2,quarter_3,quarter_4,type_organic,region_Atlanta,region_BaltimoreWashington,region_Boise,region_Boston,region_BuffaloRochester,region_Charlotte,region_Chicago,region_CincinnatiDayton,region_Columbus,region_DallasFtWorth,region_Denver,region_Detroit,region_GrandRapids,region_HarrisburgScranton,region_HartfordSpringfield,region_Houston,region_Indianapolis,region_Jacksonville,region_LasVegas,region_LosAngeles,region_Louisville,region_MiamiFtLauderdale,region_Nashville,region_NewOrleansMobile,region_NewYork,region_Orlando,region_Philadelphia,region_PhoenixTucson,region_Pittsburgh,region_Portland,region_RaleighGreensboro,region_RichmondNorfolk,region_Roanoke,region_Sacramento,region_SanDiego,region_SanFrancisco,region_Seattle,region_SouthCarolina,region_Spokane,region_StLouis,region_Syracuse,region_Tampa,AveragePrice
3210,-0.239991,-0.095385,-0.368971,-0.327777,-0.123308,-0.286169,0.465944,-0.236110,-0.164266,-0.907095,False,False,False,0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,1.07
6107,2.166901,0.486205,1.038805,0.522224,4.576195,4.902698,1.015326,0.028178,0.900074,-1.189862,False,False,False,0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,0.68
12881,-0.548822,-0.401402,-0.518188,-0.341225,-0.463676,-0.431609,-0.280115,-0.236110,0.900074,-0.341563,True,False,False,1,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,1.15
11792,-0.531430,-0.399707,-0.503961,-0.341225,-0.430131,-0.391278,-0.286088,-0.236110,0.900074,1.637802,False,False,True,1,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,1.63
3931,0.204787,0.053319,0.424518,-0.029143,0.060112,0.184313,-0.332724,-0.218445,-0.164266,-0.341563,True,False,False,0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,0.90


## Summary

- Parsed `Date` into `year`, `month` (kept numeric/ordinal) and `quarter` (one-hot encoded).
- Encoded `type` as a single binary column, `type_organic`.
- Made an explicit, justified call to **drop the 11 aggregate `region` rollups** before
  one-hot encoding, so the model trains on non-overlapping granular markets rather than a mix
  of markets and their own double-counted sums.
- One-hot encoded the remaining granular regions with `drop_first=True` to avoid the dummy
  variable trap.
- Split 80/20 into train/test with a random split, explicitly noting that a true forecasting
  application should split by date instead.
- Scaled numeric features with a scaler **fit only on the training set**, to avoid leakage.
- Saved `avocado_processed_train.csv` and `avocado_processed_test.csv` for the modeling
  notebook.

**Next up:** `03_train_test_eval.ipynb` fits and evaluates the linear regression model itself.
